In [18]:
import torch
import torch.optim as optim
from transformers import T5ForConditionalGeneration, T5Tokenizer
import time
from typing import List, Tuple
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
import sys
import os
sys.path.append(os.path.abspath(".."))
from stylistic_vector.style_metrics import style_loss
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from stylistic_vector.style_features import extract_features_batch


In [10]:
import sys
print(sys.executable)

c:\Users\fhjrj\miniconda3\envs\ml_gpu_310\python.exe


In [11]:
# Cell 2: Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model_name = "google/flan-t5-base"
print(f"Loading model: {model_name}...")
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print()

Using device: cuda
GPU: NVIDIA GeForce RTX 3080
Memory allocated: 4.37 GB
Loading model: google/flan-t5-base...
Model loaded: google/flan-t5-base
Model parameters: 247,577,856
Tokenizer vocab size: 32,000



In [12]:
# Cell 3: Define dataset class
class ConversationDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=128):
        self.data = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prefix = "Write a reply in your normal texting style:"
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        message = str(self.data.iloc[idx]['message'])
        response = str(self.data.iloc[idx]['response'])
        
        input_text = self.prefix + message
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target_encoding = self.tokenizer(
            response,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        labels = target_encoding['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(0),
            'attention_mask': input_encoding['attention_mask'].squeeze(0),
            'labels': labels.squeeze(0)
        }

In [28]:
# Cell 4: Load Your Data
train_csv_path = "../data/processed/mr_train.csv"
val_csv_path = "../data/processed/mr_val.csv"

train_dataset = ConversationDataset(train_csv_path, tokenizer)
val_dataset = ConversationDataset(val_csv_path, tokenizer)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

Train samples: 1600
Val samples: 400


In [29]:
# Cell 5: Create DataLoaders
batch_size = 8

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches: {len(val_dataloader)}")

Train batches: 200
Val batches: 50


In [20]:
# Load style stats
style_mean = np.load("../style_mean.npy")
style_std = np.load("../style_std.npy")

# Convert to torch tensors on GPU
style_mean_tensor = torch.tensor(style_mean, device=device)
style_std_tensor = torch.tensor(style_std, device=device)

# Thread pool for parallel style computation
style_pool = ThreadPoolExecutor(max_workers=4)

# Style loss function (vectorized batch version)
def compute_style_loss_batch(texts):
    """Compute style loss for a batch of texts in parallel"""
    if len(texts) == 0:
        return torch.tensor(0.0, device=device)
    
    # Extract features for all texts at once
    features = extract_features_batch(texts)  # [batch_size, feature_dim]
    features_tensor = torch.tensor(features, device=device)
    
    # Normalized MSE loss on GPU
    features_norm = (features_tensor - style_mean_tensor) / (style_std_tensor + 1e-8)
    losses = torch.mean(features_norm ** 2, dim=1)
    return torch.mean(losses)

In [15]:
def train_with_style(
    model,
    tokenizer,
    train_loader,
    val_loader,
    epochs=3,
    lr=3e-4,
    lambda_style=0.1
):
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")

        model.train()
        train_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # ---- CE LOSS ----
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            ce_loss = outputs.loss

            # ---- STYLE LOSS (NO GRAD) ----
            with torch.no_grad():
                generated_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_length=64
                )
                generated_texts = tokenizer.batch_decode(
                    generated_ids,
                    skip_special_tokens=True
                )

                style_losses = [
                    style_loss(text) for text in generated_texts
                ]
                style_loss_value = torch.tensor(
                    sum(style_losses) / len(style_losses),
                    device=device
                )

            # ---- TOTAL LOSS ----
            loss = ce_loss + lambda_style * style_loss_value

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(
                    f"Batch {batch_idx}/{len(train_loader)}, "
                    f"CE: {ce_loss.item():.4f}, "
                    f"Style: {style_loss_value.item():.4f}"
                )

        avg_train_loss = train_loss / len(train_loader)

        # ---- Validation (CE only) ----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                val_loss += outputs.loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {avg_val_loss:.4f}"
        )

    return model


In [16]:
print("Starting style-aware training...")

trained_model = train_with_style(
    model,
    tokenizer,
    train_dataloader,
    val_dataloader,
    epochs=3,
    lr=3e-4,
    lambda_style=0.1
)

print("Style-aware training complete!")

Starting style-aware training...
Epoch 1/3
Batch 0/1174, CE: 5.2933, Style: 0.9084
Batch 10/1174, CE: 4.7289, Style: 1.2039
Batch 20/1174, CE: 3.9658, Style: 0.8973
Batch 30/1174, CE: 4.6412, Style: 2.1433
Batch 40/1174, CE: 4.8114, Style: 1.4234
Batch 50/1174, CE: 3.9403, Style: 1.1886
Batch 60/1174, CE: 4.0134, Style: 0.8446
Batch 70/1174, CE: 4.1546, Style: 1.6696
Batch 80/1174, CE: 4.0936, Style: 2.3916
Batch 90/1174, CE: 4.9348, Style: 1.0955
Batch 100/1174, CE: 4.4156, Style: 1.7370
Batch 110/1174, CE: 3.6263, Style: 1.2685
Batch 120/1174, CE: 4.5776, Style: 1.8036
Batch 130/1174, CE: 4.2966, Style: 1.3327


KeyboardInterrupt: 

In [24]:
def train_with_style_fast(
    model,
    tokenizer,
    train_loader,
    val_loader,
    epochs=3,
    lr=3e-4,
    lambda_style=0.1,
    style_update_freq=5,        # Compute style every 5 batches
    generation_batch_size=8     # Generate in chunks to avoid OOM
):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        model.train()
        train_loss = 0
        ce_loss_total = 0
        style_loss_total = 0
        
        for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Training")):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # ---- CE LOSS (ALWAYS) ----
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            ce_loss = outputs.loss
            ce_loss_total += ce_loss.item()
            
            # ---- STYLE LOSS (EVERY N BATCHES) ----
            style_loss_value = torch.tensor(0.0, device=device)
            
            if batch_idx % style_update_freq == 0:
                with torch.no_grad():
                    # Generate responses in chunks for memory efficiency
                    all_generated_texts = []
                    batch_size = input_ids.size(0)
                    
                    for chunk_start in range(0, batch_size, generation_batch_size):
                        chunk_end = min(chunk_start + generation_batch_size, batch_size)
                        
                        chunk_input_ids = input_ids[chunk_start:chunk_end]
                        chunk_attention_mask = attention_mask[chunk_start:chunk_end]
                        
                        # Fast greedy generation (no beam search for speed)
                        chunk_generated_ids = model.generate(
                            input_ids=chunk_input_ids,
                            attention_mask=chunk_attention_mask,
                            max_length=64,
                            num_beams=1,        # Greedy = fastest
                            do_sample=False,
                            early_stopping=True
                        )
                        
                        chunk_texts = tokenizer.batch_decode(
                            chunk_generated_ids,
                            skip_special_tokens=True
                        )
                        all_generated_texts.extend(chunk_texts)
                    
                    # Compute style loss on entire batch (parallelized)
                    if len(all_generated_texts) > 0:
                        style_loss_value = compute_style_loss_batch(all_generated_texts)
                        style_loss_total += style_loss_value.item()
            
            # ---- TOTAL LOSS ----
            total_loss = ce_loss + lambda_style * style_loss_value
            
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Prevent exploding gradients
            optimizer.step()
            
            train_loss += total_loss.item()
            
            # Progress every 20 batches
            if batch_idx % 20 == 0:
                print(f"  Batch {batch_idx:4d}: CE={ce_loss.item():.4f}, Style={style_loss_value.item():.4f}")
        
        # ---- EPOCH STATISTICS ----
        avg_train_loss = train_loss / len(train_loader)
        avg_ce_loss = ce_loss_total / len(train_loader)
        avg_style_loss = style_loss_total / max(1, len(train_loader) // style_update_freq)
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Total Loss:    {avg_train_loss:.4f}")
        print(f"  CE Loss:       {avg_ce_loss:.4f}")
        print(f"  Style Loss:    {avg_style_loss:.4f}")
        
        # ---- VALIDATION ----
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                val_loss += outputs.loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        print(f"  Val Loss:      {avg_val_loss:.4f}")
        print("-" * 50)
    
    # Cleanup
    style_pool.shutdown()
    
    return model

In [30]:
print("Starting optimized style-aware training...")

trained_model = train_with_style_fast(
    model=model,
    tokenizer=tokenizer,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    epochs=3,
    lr=3e-4,
    lambda_style=0.1,
    style_update_freq=5,
    generation_batch_size=8
)

print("Optimized training complete!")

Starting optimized style-aware training...

Epoch 1/3


Training:   0%|          | 1/200 [00:11<39:12, 11.82s/it]

  Batch    0: CE=3.7645, Style=0.6784


Training:  10%|█         | 21/200 [02:34<24:07,  8.09s/it]

  Batch   20: CE=3.7163, Style=0.4687


...

  Batch  160: CE=2.4451, Style=0.3869


Training:  90%|█████████ | 181/200 [19:15<02:15,  7.11s/it]

  Batch  180: CE=2.4243, Style=0.5294


Training: 100%|██████████| 200/200 [21:14<00:00,  6.37s/it]



Epoch 3 Summary:
  Total Loss:    2.6386
  CE Loss:       2.6287
  Style Loss:    0.4978
  Val Loss:      3.8243
--------------------------------------------------
Optimized training complete!


In [31]:
# Cell 8: Generate Multiple Responses Function
def generate_responses(model, tokenizer, message, device, num_responses=3):
    input_text = "Reply: " + message
    input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_length=64,
            num_beams=5,
            num_return_sequences=num_responses,
            early_stopping=True
        )
        # output_ids = model.generate(
        #     input_ids=input_ids,
        #     max_new_tokens=50,
        #     num_return_sequences=num_responses,
        #     do_sample=True,
        #     temperature=0.3,  # Increased from 0.8 (more randomness)
        #     top_p=0.9,       # Nucleus sampling for diversity
        #     top_k=50,        # Limit to top 50 tokens
        #     repetition_penalty=1.2,  # Lower penalty = more generic
        #     num_beams=5,     # Add beams for better quality
        # )
        
		
    
    responses = []
    for i in range(num_responses):
        response = tokenizer.decode(output_ids[i], skip_special_tokens=True)
        responses.append(response)
    
    return responses

In [35]:
# Cell 9: Test Model
test_messages = [
    "Hey, what's up?",
    "How are you doing?",
    "See you tomorrow",
    "have you finished the report?",
    "later bro"
]

print("Testing model...")
for msg in test_messages:
    response = generate_responses(trained_model, tokenizer, msg, device)
    print(f"Input: {msg}")
    print(f"Response: {response}")
    print("-" * 40)

Testing model...
Input: Hey, what's up?
Response: ['what the hael', 'what the hael???', 'what the hael?']
----------------------------------------
Input: How are you doing?
Response: ["i'm not doing anything", "i'm doing some work right now", "i'm not doing any work right now"]
----------------------------------------
Input: See you tomorrow
Response: ['what the hael????', 'what the hael', 'what the hael going on']
----------------------------------------
Input: have you finished the report?
Response: ['no not yet', "i haven't finished", 'no i have not finished']
----------------------------------------
Input: later bro
Response: ['alright bro', "bruh i'm sorry bro", "bruh i'm not a fan of anime"]
----------------------------------------


In [37]:
# Cell 10: Save Model
save_path = "../models/style_loss_model"
trained_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Model saved to ../models/style_loss_model
